In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import netCDF4 as nc
import os
import pickle


In [2]:
import sys
sys.path.append('/home/z5297792/UNSW-MRes/MRes/SEACOFS_dataset') 
from clim_functions import find_directional_radii, rotate_uv, transect_indexer, nearest_ij, interp_3d_to_reference_depths, local_eddy_arrays, vert_doppio_dataset


In [3]:
fname = f'/srv/scratch/z3533156/26year_BRAN2020/outer_avg_01461.nc'

dataset = nc.Dataset(fname)
lon_rho = np.transpose(dataset.variables['lon_rho'], axes=(1, 0))
lat_rho = np.transpose(dataset.variables['lat_rho'], axes=(1, 0))
mask_rho = np.transpose(dataset.variables['mask_rho'], axes=(1, 0))
h = np.transpose(dataset.variables['h'], axes=(1, 0))
# f = np.transpose(dataset.variables['f'], axes=(1, 0))
angle = dataset.variables['angle'][0, 0]
z_r = np.load('/srv/scratch/z5297792/z_r.npy')
z_r = np.transpose(z_r, (1, 2, 0))
def distance(lat1, lon1, lat2, lon2):
    EARTH_RADIUS = 6357
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return EARTH_RADIUS * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
i_mid, j_mid = lon_rho.shape[0] // 2, lon_rho.shape[1] // 2
dx = distance(lat_rho[:-1, j_mid], lon_rho[:-1, j_mid],
              lat_rho[1:, j_mid], lon_rho[1:, j_mid])
dy = distance(lat_rho[i_mid, :-1], lon_rho[i_mid, :-1],
              lat_rho[i_mid, 1:], lon_rho[i_mid, 1:])
x_grid = np.insert(np.cumsum(dx), 0, 0)
y_grid = np.insert(np.cumsum(dy), 0, 0)
X_grid, Y_grid = np.meshgrid(x_grid, y_grid, indexing='ij')


In [4]:
df_eddies = pd.read_pickle('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/DOPPIO_SEACOFS_26yr_50m_vert_check/df_eddies_50m_vert_checked_processed.pkl')
                        

In [5]:
worker_id = 9   # 0,1,2,3

save_file = f'/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/DOPPIO_SEACOFS_26yr_50m_vert_check/dic_vert_parts/dic_vert_50m_vert_check_part{worker_id}.pkl'

fnames = np.sort(df_eddies['fname'].unique())

# split into 10 roughly equal chunks
fname_chunks = np.array_split(fnames, 10)

# choose which worker this script is

fnames = fname_chunks[worker_id]

target_depths = np.abs(z_r[150,150,1:]) 

if os.path.exists(save_file):
    with open(save_file, 'rb') as f:
        dic = pickle.load(f)

    completed = []

    for eddy_key, eddy_dic in dic.items():
        eddy = int(eddy_key[4:])

        for day_key in eddy_dic.keys():
            day = int(day_key[3:])
            completed.append((eddy, day))

    df_completed = pd.DataFrame(completed, columns=['Eddy', 'Day'])

    df_completed = df_completed.merge(
        df_eddies[['Eddy', 'Day', 'fname']],
        on=['Eddy', 'Day'],
        how='left'
    )

    df_completed['fnumber'] = df_completed['fname'].str[48:53].astype(int)

    last_fnumber = df_completed['fnumber'].max()

    print(f'Resuming from fnumber >= {last_fnumber}')

    fnames = [
        f for f in fnames
        if int(f[48:53]) >= last_fnumber
    ]

else:
    dic = {}
    last_fnumber = None
    print('Starting from scratch')

for fname in fnames:

    dic = vert_doppio_dataset(
        dic,
        df_eddies,
        fname,
        X_grid,
        Y_grid,
        angle,
        z_r,
        r=30,
        max_depth_levels=None,
        target_depths=target_depths
    )

    with open(save_file, 'wb') as f:
        pickle.dump(dic, f)

    print(f'Processed {fname}')

    

Starting from scratch
Processed /srv/scratch/z3533156/26year_BRAN2020/outer_avg_09771.nc
Processed /srv/scratch/z3533156/26year_BRAN2020/outer_avg_09801.nc
Processed /srv/scratch/z3533156/26year_BRAN2020/outer_avg_09831.nc
Processed /srv/scratch/z3533156/26year_BRAN2020/outer_avg_09861.nc
Processed /srv/scratch/z3533156/26year_BRAN2020/outer_avg_09891.nc
Processed /srv/scratch/z3533156/26year_BRAN2020/outer_avg_09921.nc


/home/z5297792/ESP_zonodo/functions.py:517: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, pcov = curve_fit(


Processed /srv/scratch/z3533156/26year_BRAN2020/outer_avg_09951.nc
Processed /srv/scratch/z3533156/26year_BRAN2020/outer_avg_09981.nc
Processed /srv/scratch/z3533156/26year_BRAN2020/outer_avg_10011.nc
Processed /srv/scratch/z3533156/26year_BRAN2020/outer_avg_10041.nc
Processed /srv/scratch/z3533156/26year_BRAN2020/outer_avg_10071.nc
Processed /srv/scratch/z3533156/26year_BRAN2020/outer_avg_10101.nc
Processed /srv/scratch/z3533156/26year_BRAN2020/outer_avg_10131.nc
Processed /srv/scratch/z3533156/26year_BRAN2020/outer_avg_10161.nc
Processed /srv/scratch/z3533156/26year_BRAN2020/outer_avg_10191.nc
Processed /srv/scratch/z3533156/26year_BRAN2020/outer_avg_10221.nc
Processed /srv/scratch/z3533156/26year_BRAN2020/outer_avg_10251.nc
Processed /srv/scratch/z3533156/26year_BRAN2020/outer_avg_10281.nc
Processed /srv/scratch/z3533156/26year_BRAN2020/outer_avg_10311.nc
Processed /srv/scratch/z3533156/26year_BRAN2020/outer_avg_10341.nc
Processed /srv/scratch/z3533156/26year_BRAN2020/outer_avg_1037